# Qwen3-14B + 自作 LoRA を vLLM バックエンドで推論

このノートブックは `src/llmlab/backends/vllm_backend.py` の API を利用して、Qwen3-14B（HF Hub）と自作 LoRA アダプタを組み合わせた推論を行う Google Colab 向けワークフローです。各セルの役割は次の通りです。

1. 環境構築（GitHub 取得 / 依存インストール）
2. パス・モデル設定
3. バックエンド API 読み込み
4. vLLM モデルロード
5. バッチ推論 + メトリクス計測
6. 対話モード（任意）
7. 後片付け


In [ ]:
# === 1. 環境構築: GitHub 取得と依存インストール（Google Colab 想定） ===
import os
import sys

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover - Colab 前提
    get_ipython = None  # type: ignore

def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython 環境が必要です。Colab で実行してください。")
    return ip

def clone_repo(url: str, target: str) -> None:
    if os.path.exists(target):
        print("既存リポジトリを再利用します:", target)
        return
    ip = _require_ipython()
    print("リポジトリをクローンします:", url)
    ip.system(f"git clone {url} {target}")

def pip_install(packages) -> None:
    packages = list(packages)
    if not packages:
        return
    ip = _require_ipython()
    quoted = " ".join(f'"{pkg}"' for pkg in packages)
    print("pip install:", packages)
    ip.run_line_magic("pip", f"install --upgrade {quoted}")

REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = os.environ.get("LLMLAB_REPO_DIR", DEFAULT_REPO_DIR)

clone_repo(REPO_URL, REPO_DIR)

BASE_PACKAGES = [
    "transformers>=4.56.0,<4.57.0",
    "peft>=0.17.0,<0.18.0",
    "vllm",
    "pyyaml",
]
pip_install(BASE_PACKAGES)

extra = os.environ.get("LLMLAB_EXTRA_PACKAGES")
if extra:
    pip_install(extra.split())


In [ ]:
# === 2. パス・モデル設定 ===
from pathlib import Path

REPO_ROOT = Path(REPO_DIR).resolve()
if "google.colab" in sys.modules:
    DATA_ROOT = Path("/content/drive/MyDrive/llm-lab-runtime")
else:
    DATA_ROOT = Path.cwd() / "runtime"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get("QWEN3_MODEL_NAME", "Qwen/Qwen3-14B-Instruct")
# 自作 LoRA のパス（例: Google Drive に配置）。環境変数で上書き可能です。
LORA_DIR = Path(os.environ.get("QWEN3_LORA_PATH", DATA_ROOT / "loras" / "qwen3_14b_custom"))
LORA_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"LORA_DIR: {LORA_DIR}")
print("LoRA 重みを上記ディレクトリに配置してください。存在しない場合は自動的にスキップされます。")


In [ ]:
# === 3. Python パス調整とバックエンド API の読み込み ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.llmlab.backends.vllm_backend import (
    ModelBundle,
    chat_loop,
    free_model,
    load_model,
    profile_generation,
)

print("バックエンド API をロードしました。")


In [ ]:
# === 4. vLLM モデルロード ===
cfg = {
    "model_name": MODEL_NAME,
    "tensor_parallel_size": int(os.environ.get("QWEN3_TP", 1)),
    "dtype": os.environ.get("QWEN3_DTYPE", "auto"),
    "max_model_len": int(os.environ.get("QWEN3_MAX_LEN", 4096)),
    "gpu_memory_utilization": float(os.environ.get("QWEN3_GPU_UTIL", 0.90)),
    "download_dir": str(DATA_ROOT / "model_cache"),
    "quantization": os.environ.get("QWEN3_QUANT", "none"),
    "lora_path": str(LORA_DIR) if any(LORA_DIR.glob("**/*")) else None,
    "merge_lora": bool(int(os.environ.get("QWEN3_MERGE_LORA", "0"))),
}

print("ロード設定:", cfg)
bundle: ModelBundle = load_model(cfg)
print("モデルロード完了。トークナイザ取得可否:", bool(bundle["tok"]))
print("ロード時間(秒):", bundle["cfg"].get("_timings", {}))


In [ ]:
# === 5. バッチ推論 & メトリクス計測 ===
import pandas as pd

prompts = [
    "以下の仕様を要約してください:\n- 多段 LoRA でチューニングした Qwen3-14B\n- 連携するエッジ端末向けアプリへの適用",
    "LoRA 適用済みモデルとして、顧客向け FAQ 生成ボットの導入メリットを3点で説明してください。",
]

outputs, metrics = profile_generation(
    bundle,
    prompts,
    max_new_tokens=int(os.environ.get("QWEN3_MAX_NEW", 256)),
    temperature=float(os.environ.get("QWEN3_TEMP", 0.7)),
    top_p=float(os.environ.get("QWEN3_TOP_P", 0.9)),
    repetition_penalty=float(os.environ.get("QWEN3_REP_PEN", 1.05)),
    stop=["\nUser:"],
)

display(pd.DataFrame({"prompt": prompts, "output": outputs}))
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


In [ ]:
# === 6. 対話モード（任意） ===
print("対話を開始します。終了コマンド: /exit")
try:
    chat_loop(
        bundle,
        system_prompt="あなたは Qwen3-14B ベースのアシスタントです。",
        stop_phrases=["/exit", ":q"],
        max_new_tokens=256,
        temperature=0.7,
    )
finally:
    print("対話モードを終了しました。")


In [ ]:
# === 7. 後片付け ===
free_model(bundle)
print("メモリを解放しました。")
